# Odlar iş masası: təlim

Bütün Odlar modelləri bu bir notebook ilə öyrədilir. Fərq yalnız config faylındadır (`configs/` qovluğu).

1. **Runtime → Change runtime type → T4 GPU** → Save (Kaggle-da: Settings → Accelerator → GPU T4)
2. Aşağıda `CONFIG` sətrində modeli seç
3. **Runtime → Run all**. Google Drive icazəsi soruşanda icazə ver
4. Sessiya kəsilsə, yenidən **Run all**: təlim qaldığı yerdən davam edir

İstəyə bağlı: checkpoint-lərin Hugging Face-də də saxlanması üçün config-də `hub_repo` var.
Onda Colab-da solda açar işarəsinə (Secrets) bas, `HF_TOKEN` adı ilə öz write token-ini əlavə et.

In [ ]:
# 1) Ayarlar
CONFIG = "configs/odlar15.yaml"   # odlar1.yaml, odlar15.yaml, odlar2.yaml, tiny.yaml
EXTRA  = []                       # məs. ["train.max_steps=20000"]

In [ ]:
# 2) Hazırlıq: GPU, Drive, kod
import os, subprocess, yaml
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "GPU tapılmadı!")
name = os.path.splitext(os.path.basename(CONFIG))[0]
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/Odlar/runs/{name}"
except ImportError:
    OUT = os.path.abspath(f"/kaggle/working/runs/{name}" if os.path.exists("/kaggle") else f"runs/{name}")
REPO = "/content/odlar-repo" if os.path.exists("/content") else (
    "/kaggle/working/odlar-repo" if os.path.exists("/kaggle") else os.path.abspath("odlar-repo"))
if not os.path.exists(REPO):
    !git clone -q https://github.com/krmhuseyn-sys/odlar.git {REPO}
else:
    !cd {REPO} && git pull -q
%cd {REPO}
!pip install -q -r requirements-train.txt
# HF_TOKEN (istəyə bağlı): Colab/Kaggle Secrets-dən götürülür, təlim prosesinə ötürülür
for get in (lambda: __import__("google.colab", fromlist=["userdata"]).userdata.get("HF_TOKEN"),
            lambda: __import__("kaggle_secrets").UserSecretsClient().get_secret("HF_TOKEN")):
    try:
        os.environ["HF_TOKEN"] = get(); print("HF_TOKEN tapıldı"); break
    except Exception:
        pass
print("Nəticələr:", OUT)

In [ ]:
# 3) Təlim (data yoxdursa, əvvəlcə hazırlanır)
sets = " ".join([f"out_dir={OUT}"] + EXTRA)
!python -m odlar.train {CONFIG} --set {sets}

In [ ]:
# 4) Sınaq: mətn yazdır və standart testlər
!python -m odlar.generate {OUT} "Qış gecəsi idi" -n 2 --tokens 80
!python -m odlar.evaluate {OUT} --out {OUT}/eval.json

In [ ]:
# 5) Brauzer üçün ONNX (sayta qoymaq üçün). Fayllar OUT/web qovluğuna yazılır
!python -m odlar.export {OUT} --name {name} --to {OUT}/web